In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/otfs-channel-estiamtion-v3/Hcap_val_sample0_ant0.csv
/kaggle/input/otfs-channel-estiamtion-v3/Htrue_sample0_ant0.csv
/kaggle/input/otfs-channel-estiamtion-v3/channel_est_preload_all_chunks.pth
/kaggle/input/otfs-channel-estiamtion-v3/Hcap_thr_val_sample0_ant0.csv
/kaggle/input/otfs-channel-estiamtion-v3/__results__.html
/kaggle/input/otfs-channel-estiamtion-v3/Hcap_sample0_ant0.csv
/kaggle/input/otfs-channel-estiamtion-v3/__notebook__.ipynb
/kaggle/input/otfs-channel-estiamtion-v3/Htrue_val_sample0_ant0.csv
/kaggle/input/otfs-channel-estiamtion-v3/__output__.json
/kaggle/input/otfs-channel-estiamtion-v3/custom.css
/kaggle/input/otfs-channel-estiamtion-v3/eval_by_snr/eval_summary_by_snr.csv
/kaggle/input/otfs-channel-estiamtion-v3/eval_by_snr/eval_summary_by_chunk.csv
/kaggle/input/phi-256-8/Phi_pilotWeighted_padded.mat
/kaggle/input/dataset-npy-258-8-full-npy/dataset-npy-258-8-full-npy/chunk_001_yDD.npy
/kaggle/input/dataset-npy-258-8-full-npy/dataset-npy-258-8-full-npy/c

In [2]:
os.environ['PRELOAD_NPY_DIR'] = "/kaggle/input/dataset-npy-258-8-full-npy/dataset-npy-258-8-full-npy"
os.environ['PRELOAD_FILES_PER_GROUP'] = "4"
os.environ['PRELOAD_GROUPS_IN_MEMORY'] = "2"

In [ ]:
# -------------------------
# Preload-all-chunks training script 
import os, bisect, time, math
import numpy as np
import scipy.io as sio
try:
    import h5py
    _HAS_H5PY = True
except Exception:
    h5py = None
    _HAS_H5PY = False

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader


DATA_DIR = "/kaggle/input/dataset-npy-258-8-full-npy/dataset-npy-258-8-full-npy"   # folder with chunk_*.mat
PHI_PATH = "/kaggle/input/phi-256-8/Phi_pilotWeighted_padded.mat"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 20
BATCH_SIZE = 16
NUM_WORKERS = 4  # Kaggle: 4 CPU cores available; adjust if you want
PIN_MEMORY = False
LR = 1e-3


def _to_numpy_complex(arr):
    """Convert array (possibly matlab compound dtype) to numpy complex64."""
    arr = np.asarray(arr)
    if np.iscomplexobj(arr):
        return arr.astype(np.complex64)
    if arr.dtype.names is not None:
        names = tuple(n.lower() for n in arr.dtype.names)
        if 'real' in names and 'imag' in names:
            real = arr['real'].astype(np.float32)
            imag = arr['imag'].astype(np.float32)
            return (real + 1j*imag).astype(np.complex64)
    return arr.astype(np.complex64)

def _load_mat_var(path, varname):
    """Load a variable from .mat. Use scipy for v7.2 and h5py for v7.3 fallback."""
    try:
        mat = sio.loadmat(path, variable_names=[varname])
        if varname in mat:
            return mat[varname]
    except NotImplementedError:
        pass

    if not _HAS_H5PY:
        raise RuntimeError("MAT v7.3 file encountered but h5py is not available.")
    with h5py.File(path, 'r') as f:
        if varname in f:
            return f[varname][:]
        for k in f.keys():
            if varname in k:
                return f[k][:]
    raise KeyError(f"Could not locate {varname} in {path}")

# Dataset: PRELOAD all chunks 
from collections import OrderedDict
import bisect
import numpy as np
import scipy.io as sio
import os
import math
import torch
from torch.utils.data import Dataset



class PreloadedSamplesFirstDataset(Dataset):
    
    def __init__(self, data_dir, split='train', split_ratio=0.9, verbose=True):
        self.data_dir = data_dir
        self.verbose = verbose

        try:
            self.files_per_group = int(os.environ.get('PRELOAD_FILES_PER_GROUP', 4))
        except Exception:
            self.files_per_group = 4
        try:
            self.groups_in_memory = int(os.environ.get('PRELOAD_GROUPS_IN_MEMORY', 1))
        except Exception:
            self.groups_in_memory = 1
        if self.files_per_group < 1: self.files_per_group = 1
        if self.groups_in_memory < 1: self.groups_in_memory = 1

        self.npy_dir = os.environ.get('PRELOAD_NPY_DIR', self.data_dir)
        self.npy_dir = os.path.abspath(self.npy_dir)

        if not os.path.isdir(self.npy_dir):
            raise RuntimeError(f"PRELOAD_NPY_DIR {self.npy_dir!r} does not exist or is not a directory.")
        hadd_files = sorted([f for f in os.listdir(self.npy_dir) if f.endswith('_HADD.npy')])
        bases = []
        for f in hadd_files:
            base = f[:-9]  # strip "_HADD.npy"
            yname = base + '_yDD.npy'
            if os.path.exists(os.path.join(self.npy_dir, yname)):
                bases.append(base)
            else:
                if self.verbose:
                    print(f"[Dataset] Warning: found {f} but missing {yname}; skipping.")
        if len(bases) == 0:
            raise RuntimeError(f"No per-chunk '_HADD.npy'/_yDD.npy pairs found in {self.npy_dir!r}")
        self._bases_all = bases

        if self.verbose:
            print(f"[Dataset] found {len(self._bases_all)} chunk bases in {self.npy_dir!r}")
            print(f"[Dataset] files_per_group={self.files_per_group}, groups_in_memory={self.groups_in_memory}")

        try:
            seed = os.environ.get('PRELOAD_CHUNK_SEED', None)
            rng = np.random.default_rng(int(seed)) if seed is not None else np.random.default_rng()
        except Exception:
            rng = np.random.default_rng()
        order = list(range(len(self._bases_all)))
        rng.shuffle(order)
        self.files = [self._bases_all[i] for i in order]
        if self.verbose:
            print(f"[Dataset] randomized chunk order (first 8 indices): {order[:8]}")

        self.samples_per_file = []
        for base in self.files:
            y_path = os.path.join(self.npy_dir, base + '_yDD.npy')
            # assume file exists (we filtered earlier); use mmap to read shape without loading
            try:
                y_mm = np.load(y_path, mmap_mode='r')
                self.samples_per_file.append(int(y_mm.shape[0]))
            except Exception as e:
                raise RuntimeError(f"Error reading {y_path} to determine sample count: {e}")

        self.total_samples = int(sum(self.samples_per_file))
        if self.verbose:
            print(f"[Dataset] total_samples across chunks = {self.total_samples}")
            print(f"[Dataset] samples_per_file (first 10) = {self.samples_per_file[:10]}")

        self.cum_counts = np.cumsum([0] + list(self.samples_per_file))

        indices = list(range(self.total_samples))
        split_idx = int(np.floor(self.total_samples * split_ratio))
        if self.total_samples >= 2:
            split_idx = min(max(1, split_idx), self.total_samples - 1)
        else:
            split_idx = self.total_samples

        chosen_global = indices[:split_idx] if split == 'train' else indices[split_idx:]
        if len(chosen_global) == 0:
            if self.verbose:
                print(f"[Dataset] WARNING: split '{split}' produced 0 samples; using all samples.")
            chosen_global = indices

        chunked_indices = []
        for file_idx in range(len(self.files)):
            start = int(self.cum_counts[file_idx]); end = int(self.cum_counts[file_idx + 1])
            chunk_inds = [g for g in chosen_global if (g >= start and g < end)]
            if split == 'train':
                rng.shuffle(chunk_inds)
            chunked_indices.extend(chunk_inds)
        self.global_indices = chunked_indices
        if self.verbose:
            print(f"[Dataset] split='{split}', split_ratio={split_ratio}, samples={len(self.global_indices)} (chunk-local ordering)")

        self.numDD = None; self.Nt = None; self.N = None; self.M = None

        self._group_cache = OrderedDict()

        if len(self.files) > 0:
            first_group_start = self._group_start_for_file(0)
            self._ensure_group_loaded(first_group_start)
            first_entry = next(iter(self._group_cache.values()))
            if 'y_list' in first_entry and len(first_entry['y_list'])>0:
                self.numDD = int(first_entry['y_list'][0].shape[1])
            if 'hadd_list' in first_entry and len(first_entry['hadd_list'])>0:
                hadd0 = first_entry['hadd_list'][0]
                if hadd0.ndim != 4:
                    raise RuntimeError(f"HADD expected 4D (S,Nt,N,M) but got {hadd0.shape}")
                self.Nt = int(hadd0.shape[1]); self.N = int(hadd0.shape[2]); self.M = int(hadd0.shape[3])
            if self.verbose:
                print(f"[Dataset] Inferred dims from first cached group: numDD={self.numDD}, M={self.M}, N={self.N}, Nt={self.Nt}")

    def __len__(self):
        return len(self.global_indices)

    def _group_start_for_file(self, file_idx):
        return (file_idx // self.files_per_group) * self.files_per_group

    def _files_in_group(self, group_start):
        return list(range(group_start, min(group_start + self.files_per_group, len(self.files))))

    def _npy_paths_for_base(self, base):
        hadd_npy = os.path.join(self.npy_dir, base + '_HADD.npy')
        y_npy = os.path.join(self.npy_dir, base + '_yDD.npy')
        return hadd_npy, y_npy

    def _ensure_group_loaded(self, group_start):
        if group_start in self._group_cache:
            self._group_cache.move_to_end(group_start, last=True)
            return self._group_cache[group_start]

        files_idx = self._files_in_group(group_start)
        y_list = []; hadd_list = []; sizes = []
        all_memmap = True

        for fi in files_idx:
            base = self.files[fi]
            hadd_path, y_path = self._npy_paths_for_base(base)

            try:
                y_mem = np.load(y_path, mmap_mode='r')
                hadd_mem = np.load(hadd_path, mmap_mode='r')
            except Exception as e:
                raise RuntimeError(f"Error loading memmap pair for base {base}: {e}")

            if y_mem.dtype != np.complex64:
                if self.verbose:
                    print(f"[Dataset] Warning: casting {y_path} from {y_mem.dtype} -> complex64 (loading into memory).")
                y_arr = y_mem.astype(np.complex64)
                all_memmap = False
            else:
                y_arr = y_mem

            if hadd_mem.dtype != np.complex64:
                if self.verbose:
                    print(f"[Dataset] Warning: casting {hadd_path} from {hadd_mem.dtype} -> complex64 (loading into memory).")
                hadd_arr = hadd_mem.astype(np.complex64)
                all_memmap = False
            else:
                hadd_arr = hadd_mem

            y_list.append(y_arr)
            hadd_list.append(hadd_arr)
            sizes.append(int(y_arr.shape[0]))

        entry = {'bases': files_idx, 'sizes': sizes, 'y_list': y_list, 'hadd_list': hadd_list, 'is_memmap': all_memmap}
        self._group_cache[group_start] = entry
        self._group_cache.move_to_end(group_start, last=True)

        while len(self._group_cache) > self.groups_in_memory:
            evicted_group_start, _ = self._group_cache.popitem(last=False)
            
        return entry

    def _global_to_file_local(self, gidx):
        file_idx = bisect.bisect_right(self.cum_counts, gidx) - 1
        local_idx = int(gidx - self.cum_counts[file_idx])
        return file_idx, local_idx

    def __getitem__(self, idx):
        gidx = self.global_indices[idx]
        file_idx, local_idx = self._global_to_file_local(gidx)

        group_start = self._group_start_for_file(file_idx)
        entry = self._ensure_group_loaded(group_start)

        files_in_group = entry['bases']
        file_pos = None
        for i, fi in enumerate(files_in_group):
            if fi == file_idx:
                file_pos = i
                break
        if file_pos is None:
            raise RuntimeError("Internal error: file_idx not found in group entry")

        y_arr = entry['y_list'][file_pos]       # (S_file, numDD)
        hadd_arr = entry['hadd_list'][file_pos] # (S_file, Nt, N, M)

        y_row = y_arr[local_idx, :].astype(np.complex64)
        hadd_sample = hadd_arr[local_idx, :, :, :].astype(np.complex64)

        Hadd = np.transpose(hadd_sample, (2, 1, 0)).astype(np.complex64)
        return y_row, Hadd


def derive_pilot_coords_from_Phi(Phi, M, N, Nt):
    Phi = np.asarray(Phi)
    numDD, Ptot = Phi.shape
    if numDD != M * N:
        raise RuntimeError(f"Phi numDD ({numDD}) != M*N ({M}*{N})")
    if Ptot % Nt != 0:
        raise RuntimeError(f"Phi Ptot ({Ptot}) not divisible by Nt ({Nt})")
    P = Ptot // Nt
    pilot_rows = np.empty(P, dtype=int)
    pilot_cols = np.empty(P, dtype=int)
    base = 0
    for p in range(P):
        col = Phi[:, base + p]
        idx = int(np.argmax(np.abs(col)))
        r, c = np.unravel_index(idx, (M, N), order='C')
        pilot_rows[p] = r
        pilot_cols[p] = c
    return pilot_rows, pilot_cols

def scatter_batch_feats_to_X(feats, Nt, M, N, pilot_rows, pilot_cols):
    
    Ptot, B = feats.shape
    expected_full = Nt * M * N
    if Ptot == expected_full:
        X = feats.T.reshape((B, Nt, M, N), order='C')
        return X
    if Ptot % Nt != 0:
        raise RuntimeError("Feats length not divisible by Nt and not full-grid.")
    P = Ptot // Nt
    pr = np.asarray(pilot_rows).ravel()
    pc = np.asarray(pilot_cols).ravel()
    if pr.size != P or pc.size != P:
        raise RuntimeError("pilot_rows/pilot_cols length mismatch P")
    feat3 = feats.reshape((Nt, P, B), order='C')  # (Nt, P, B)
    X = np.zeros((B, Nt, M, N), dtype=np.complex64)
    for t in range(Nt):
        vals = feat3[t]            # (P, B)
        X[:, t, pr, pc] = vals.T   # (B,P) assign into (B,P)
    return X

def make_batch_collate_fn(Phi, M, N, Nt, pilot_rows, pilot_cols):
    Phi = np.asarray(Phi).astype(np.complex64)
    def collate(batch):
        B = len(batch)
        Ys = np.stack([b[0] for b in batch], axis=1).astype(np.complex64)   # (numDD, B)
        Hs = np.stack([b[1] for b in batch], axis=0).astype(np.complex64)   # (B, M, N, Nt)
        Feats = Phi.conj().T.dot(Ys)   # (Ptot, B)
        Xc = scatter_batch_feats_to_X(Feats, Nt, M, N, pilot_rows, pilot_cols)   # (B, Nt, M, N)
        X_real = Xc.real.astype(np.float32); X_imag = Xc.imag.astype(np.float32)
        X_batch = np.concatenate([X_real, X_imag], axis=1)  # (B, 2*Nt, M, N)
        Yc = np.transpose(Hs, (0, 3, 1, 2))   # (B, Nt, M, N)
        Y_real = Yc.real.astype(np.float32); Y_imag = Yc.imag.astype(np.float32)
        Y_batch = np.concatenate([Y_real, Y_imag], axis=1)  # (B, 2*Nt, M, N)
        return torch.from_numpy(X_batch).float(), torch.from_numpy(Y_batch).float()
    return collate


class Residual2D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        identity = x
        out = self.conv1(x); out = self.bn1(out); out = self.relu(out)
        out = self.conv2(out); out = self.bn2(out)
        out = out + identity
        out = self.relu(out)
        return out

class ChannelEst2DNet(nn.Module):
    def __init__(self, in_channels, Nt, M, N):
        super().__init__()
        self.conv_in = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn_in = nn.BatchNorm2d(32)
        self.relu = nn.ReLU(inplace=True)
        self.conv_mid = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn_mid = nn.BatchNorm2d(64)
        self.res1 = Residual2D(64); self.res2 = Residual2D(64); self.res3 = Residual2D(64)
        self.conv_red = nn.Conv2d(64, 32, kernel_size=3, padding=1)
        self.bn_red = nn.BatchNorm2d(32)
        self.conv_out = nn.Conv2d(32, 2 * Nt, kernel_size=3, padding=1)
    def forward(self, x):
        out = self.relu(self.bn_in(self.conv_in(x)))
        out = self.relu(self.bn_mid(self.conv_mid(out)))
        out = self.res1(out); out = self.res2(out); out = self.res3(out)
        out = self.relu(self.bn_red(self.conv_red(out)))
        out = self.conv_out(out)
        return out


class WeightedComplexMSELoss(nn.Module):
    
    def __init__(self, threshold_ratio=0.01, gamma=0.1, l1_lambda=1e-2,
                 soft_thresh_ratio=0.01, eps=1e-12):
        super().__init__()
        self.threshold_ratio = float(threshold_ratio)
        self.gamma = float(gamma)
        self.l1_lambda = float(l1_lambda)
        self.soft_thresh_ratio = float(soft_thresh_ratio)
        self.eps = float(eps)

    def forward(self, pred, target):
        B, C, M, N = pred.shape
        Nt = C // 2

        pred_real = pred[:, :Nt, :, :]
        pred_imag = pred[:, Nt:, :, :]
        targ_real = target[:, :Nt, :, :]
        targ_imag = target[:, Nt:, :, :]

        targ_mag_sq = torch.sum(targ_real * targ_real + targ_imag * targ_imag, dim=1)  # (B, M, N)
        targ_mag = torch.sqrt(targ_mag_sq + self.eps)                                  # (B, M, N)

        max_mag = torch.amax(targ_mag.view(B, -1), dim=1).view(B, 1, 1) + self.eps
        thresh = self.threshold_ratio * max_mag   # (B,1,1) -- weighting threshold (unchanged)

        tau = (self.soft_thresh_ratio * max_mag).view(B, 1, 1, 1)  # broadcastable to (B, Nt, M, N)

        pred_mag_elem_sq = pred_real * pred_real + pred_imag * pred_imag  # (B, Nt, M, N)
        pred_mag_elem = torch.sqrt(pred_mag_elem_sq + self.eps)

        shrink = torch.relu(pred_mag_elem - tau)

        denom = pred_mag_elem + self.eps
        scale = shrink / denom   # (B, Nt, M, N), 0 where pred_mag_elem <= tau

        pred_real_thr = pred_real * scale
        pred_imag_thr = pred_imag * scale

        weights = torch.where(targ_mag >= thresh,
                              torch.ones_like(targ_mag),
                              torch.full_like(targ_mag, self.gamma))  # (B,M,N)
        weights = weights.unsqueeze(1)  # (B,1,M,N) broadcast over antennas

        se = (pred_real_thr - targ_real) ** 2 + (pred_imag_thr - targ_imag) ** 2  # (B, Nt, M, N)
        se_sum = torch.sum(se, dim=1)  # (B, M, N) sum across antennas

        weighted_se = weights * se_sum
        weighted_mse = torch.mean(weighted_se)   # scalar

        pred_mag_sq_raw = torch.sum(pred_real * pred_real + pred_imag * pred_imag, dim=1)  # (B,M,N)
        pred_mag_raw = torch.sqrt(pred_mag_sq_raw + self.eps)                              # (B,M,N)

        small_mask = (targ_mag < thresh).to(dtype=pred_mag_raw.dtype)  # (B,M,N)

        masked_l1 = torch.mean(pred_mag_raw * small_mask)

        total_loss = weighted_mse + self.l1_lambda * masked_l1
        return total_loss



In [ ]:

def main():
    print("Device:", DEVICE)
    ds_train = PreloadedSamplesFirstDataset(DATA_DIR, split='train', split_ratio=0.9, verbose=True)
    ds_val   = PreloadedSamplesFirstDataset(DATA_DIR, split='val',   split_ratio=0.9, verbose=True)

    print("Train samples:", len(ds_train), "Val samples:", len(ds_val))

    Phi_raw = _load_mat_var(PHI_PATH, 'Phi')
    Phi_tmp = _to_numpy_complex(Phi_raw)
    if Phi_tmp.shape[0] == ds_train.numDD:
        Phi = Phi_tmp
    elif Phi_tmp.shape[1] == ds_train.numDD:
        Phi = Phi_tmp.T.copy()
        print("[Main] Transposed loaded Phi to match numDD axis.")
    else:
        raise RuntimeError(f"Loaded Phi shape {Phi_tmp.shape} incompatible with dataset.numDD={ds_train.numDD}")

    Ptot = Phi.shape[1]
    print(f"[Main] Loaded Phi shape (numDD, Ptot) = {Phi.shape}. Ptot={Ptot}")

    pilot_rows, pilot_cols = derive_pilot_coords_from_Phi(Phi, ds_train.M, ds_train.N, ds_train.Nt)
    print(f"[Main] Derived pilot count P = {pilot_rows.size}. Example pilot (r,c) = ({pilot_rows[0]},{pilot_cols[0]})")

    collate_fn = make_batch_collate_fn(Phi, ds_train.M, ds_train.N, ds_train.Nt, pilot_rows, pilot_cols)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)
    loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)

    sampleX, sampleY = next(iter(loader_train))
    print("Sample X shape (B,C,M,N):", sampleX.shape)
    print("Sample Y shape (B,C,M,N):", sampleY.shape)

    in_channels = sampleX.shape[1]
    Nt = ds_train.Nt; M = ds_train.M; N = ds_train.N
    if in_channels != 2 * Nt:
        raise RuntimeError(f"in_channels ({in_channels}) != 2*Nt ({2*Nt}). Check mapping or model.")
    model = ChannelEst2DNet(in_channels=in_channels, Nt=Nt, M=M, N=N).to(DEVICE)

    
    criterion = WeightedComplexMSELoss(threshold_ratio=0.05, gamma=0.05, l1_lambda=1e-2).to(DEVICE)

    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_val = float('inf')
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0; n = 0; t0 = time.time()
        for X, Y in loader_train:
            X = X.to(DEVICE); Y = Y.to(DEVICE)
            optimizer.zero_grad()
            Yp = model(X)
            loss = criterion(Yp, Y)  
            loss.backward()
            optimizer.step()
            b = X.size(0)
            running_loss += loss.item() * b
            n += b
        train_loss = running_loss / n if n>0 else 0.0
        t_elapsed = time.time() - t0

        model.eval()
        running_val = 0.0; nv = 0
        total_err = 0.0
        total_pow = 0.0
        with torch.no_grad():
            for Xv, Yv in loader_val:
                Xv = Xv.to(DEVICE); Yv = Yv.to(DEVICE)
                Ypv = model(Xv)                      

                lossv_tensor = criterion(Ypv, Yv)
                lossv = float(lossv_tensor.item())
                running_val += lossv * Xv.size(0)
                nv += Xv.size(0)

                Ypv_np = Ypv.cpu().numpy()   # (B, 2*Nt, M, N)
                Yv_np  = Yv.cpu().numpy()
                B, C, Mm, Nn = Yv_np.shape
                Nt_here = C // 2
                Ypv_c = Ypv_np[:, :Nt_here, ...] + 1j * Ypv_np[:, Nt_here:, ...]   # (B, Nt, M, N)
                Yv_c  = Yv_np[:, :Nt_here, ...]  + 1j * Yv_np[:, Nt_here:, ...]
                err_per_sample = np.sum(np.abs(Ypv_c - Yv_c)**2, axis=(1,2,3))
                pow_per_sample = np.sum(np.abs(Yv_c)**2, axis=(1,2,3))
                pow_per_sample = np.where(pow_per_sample == 0.0, 1e-12, pow_per_sample)
                total_err += float(np.sum(err_per_sample))
                total_pow += float(np.sum(pow_per_sample))

        val_loss = running_val / nv if nv>0 else float('inf')
        if total_pow > 0:
            val_nmse_linear = total_err / total_pow
            val_nmse_db = 10.0 * math.log10(val_nmse_linear)
        else:
            val_nmse_linear = float('inf')
            val_nmse_db = float('inf')

        print(f"Epoch {epoch+1}/{EPOCHS}: train_loss={train_loss:.6e} val_loss={val_loss:.6e} val_NMSE={val_nmse_db:.3f} dB time={t_elapsed:.1f}s")

        if val_loss < best_val:
            best_val = val_loss
            torch.save({'epoch': epoch+1, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict()},
                       "channel_est_preload_all_chunks.pth")
            print(f"Saved best model (val={best_val:.6e})")

if __name__ == "__main__":
    main()

Device: cuda
[Dataset] found 8 chunk bases in '/kaggle/input/dataset-npy-258-8-full-npy/dataset-npy-258-8-full-npy'
[Dataset] files_per_group=4, groups_in_memory=2
[Dataset] randomized chunk order (first 8 indices): [6, 5, 1, 0, 7, 4, 2, 3]
[Dataset] total_samples across chunks = 40000
[Dataset] samples_per_file (first 10) = [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
[Dataset] split='train', split_ratio=0.9, samples=36000 (chunk-local ordering)
[Dataset] Inferred dims from first cached group: numDD=2048, M=256, N=8, Nt=16
[Dataset] found 8 chunk bases in '/kaggle/input/dataset-npy-258-8-full-npy/dataset-npy-258-8-full-npy'
[Dataset] files_per_group=4, groups_in_memory=2
[Dataset] randomized chunk order (first 8 indices): [1, 3, 6, 7, 5, 2, 0, 4]
[Dataset] total_samples across chunks = 40000
[Dataset] samples_per_file (first 10) = [5000, 5000, 5000, 5000, 5000, 5000, 5000, 5000]
[Dataset] split='val', split_ratio=0.9, samples=4000 (chunk-local ordering)
[Dataset] Inferred dims fro

In [ ]:
import os
import numpy as np
import torch
import pandas as pd

ckpt_path = "/kaggle/working/channel_est_preload_all_chunks.pth"   # your saved checkpoint
sample_idx = 0    # index in the validation split (0..len(ds_val)-1)
antenna = 0       # antenna index to inspect (0..Nt-1)
out_dir = "/kaggle/working"  
print_everything = True

os.makedirs(out_dir, exist_ok=True)

ds_val = PreloadedSamplesFirstDataset(DATA_DIR, split='val', split_ratio=0.9, verbose=False)

Phi_raw = _load_mat_var(PHI_PATH, 'Phi')
Phi_tmp = _to_numpy_complex(Phi_raw)
if Phi_tmp.shape[0] == ds_val.numDD:
    Phi = Phi_tmp
elif Phi_tmp.shape[1] == ds_val.numDD:
    Phi = Phi_tmp.T.copy()
else:
    raise RuntimeError("Phi shape mismatch vs dataset.numDD")

pilot_rows, pilot_cols = derive_pilot_coords_from_Phi(Phi, ds_val.M, ds_val.N, ds_val.Nt)
collate_fn = make_batch_collate_fn(Phi, ds_val.M, ds_val.N, ds_val.Nt, pilot_rows, pilot_cols)

if not os.path.exists(ckpt_path):
    raise FileNotFoundError(f"Checkpoint not found at: {ckpt_path}")
ck = torch.load(ckpt_path, map_location=DEVICE)
if isinstance(ck, dict) and 'model_state' in ck:
    state = ck['model_state']
elif isinstance(ck, dict) and 'model_state_dict' in ck:
    state = ck['model_state_dict']
elif isinstance(ck, dict) and 'state_dict' in ck:
    state = ck['state_dict']
else:
    state = ck  

model = ChannelEst2DNet(in_channels=2*ds_val.Nt, Nt=ds_val.Nt, M=ds_val.M, N=ds_val.N).to(DEVICE)
try:
    model.load_state_dict(state)
except Exception:
    new = {}
    for k,v in state.items():
        nk = k[len("module."): ] if k.startswith("module.") else k
        new[nk] = v
    model.load_state_dict(new)
model.eval()

if sample_idx < 0 or sample_idx >= len(ds_val):
    raise ValueError(f"sample_idx should be in [0, {len(ds_val)-1}] for validation split")
y, Hadd_true = ds_val[sample_idx]   # Hadd_true: (M, N, Nt) complex numpy

X_batch, Y_batch = collate_fn([(y, Hadd_true)])
with torch.no_grad():
    Xb = X_batch.to(DEVICE)
    Yp = model(Xb).cpu().numpy()   # (1, 2*Nt, M, N)
Yt = Y_batch.numpy()              # (1, 2*Nt, M, N)

B, C, M, N = Yp.shape
Nt_here = C // 2
Yp_complex = Yp[0, :Nt_here, ...] + 1j * Yp[0, Nt_here:, ...]   # (Nt, M, N)
Yt_complex = Yt[0, :Nt_here, ...]  + 1j * Yt[0, Nt_here:, ...]  # (Nt, M, N)

Hadd_cap = np.transpose(Yp_complex, (1, 2, 0))   # (M, N, Nt) complex numpy
Hadd_ref = np.transpose(Yt_complex, (1, 2, 0))   # (M, N, Nt) complex numpy (should equal Hadd_true)

if antenna < 0 or antenna >= Nt_here:
    raise ValueError(f"antenna must be in [0, {Nt_here-1}]")

Hcap_ant = Hadd_cap[:, :, antenna]   # shape (M, N), complex
Htrue_ant = Hadd_true[:, :, antenna] 
Href_ant = Hadd_ref[:, :, antenna]   # collated true (should match Htrue_ant)

def complex_to_str_grid(mat):
    Mloc, Nloc = mat.shape
    grid = np.empty((Mloc, Nloc), dtype=object)
    for i in range(Mloc):
        for j in range(Nloc):
            re = float(mat[i,j].real)
            im = float(mat[i,j].imag)
            sign = '+' if im >= 0 else '-'
            grid[i,j] = f"{re:.6e}{sign}{abs(im):.6e}j"
    return grid

cap_grid = complex_to_str_grid(Hcap_ant)
true_grid = complex_to_str_grid(Htrue_ant)

cap_path = os.path.join(out_dir, f"Hcap_sample{sample_idx}_ant{antenna}.csv")
true_path = os.path.join(out_dir, f"Htrue_sample{sample_idx}_ant{antenna}.csv")

pd.DataFrame(cap_grid).to_csv(cap_path, index=False, header=False)
pd.DataFrame(true_grid).to_csv(true_path, index=False, header=False)

print(f"Saved predicted CSV: {cap_path}")
print(f"Saved true CSV:      {true_path}")

if print_everything:
    print(f"\nSample idx = {sample_idx}, dataset val size = {len(ds_val)}")
    print(f"Shapes: Hadd_cap {Hadd_cap.shape}, Hadd_ref {Hadd_ref.shape} (M,N,Nt)")
    print(f"Nt={Nt_here}, M={ds_val.M}, N={ds_val.N}\n")
    np.set_printoptions(precision=4, suppress=True)
    print(f"Top-left 6x6 of Hadd_cap (complex) for antenna {antenna}:")
    print(Hcap_ant[:6, :6])
    print()
    print(f"Top-left 6x6 of Hadd_ref (collated true) for antenna {antenna}:")
    print(Href_ant[:6, :6])
    print()
    print("Saved CSVs contain entries formatted like: real+imagj (scientific notation).")

Saved predicted CSV: /kaggle/working/Hcap_sample0_ant0.csv
Saved true CSV:      /kaggle/working/Htrue_sample0_ant0.csv

Sample idx = 0, dataset val size = 4000
Shapes: Hadd_cap (256, 8, 16), Hadd_ref (256, 8, 16) (M,N,Nt)
Nt=16, M=256, N=8

Top-left 6x6 of Hadd_cap (complex) for antenna 0:
[[-0.0234+0.0741j -0.0672+0.0481j -0.074 +0.0118j -0.0588-0.0284j
  -0.0021-0.0555j  0.05  -0.0303j]
 [-0.0282+0.1448j -0.0266+0.1483j -0.02  +0.1468j -0.0229+0.1399j
  -0.0247+0.1323j -0.0229+0.1272j]
 [-0.0077+0.5493j -0.0084+0.545j  -0.0142+0.5277j -0.034 +0.5187j
  -0.0327+0.516j  -0.0335+0.4818j]
 [ 0.04  -0.2216j  0.0269-0.2085j  0.0185-0.2075j  0.0247-0.2158j
   0.0268-0.2083j  0.0353-0.2002j]
 [ 0.0212-0.0979j  0.0216-0.1086j  0.0345-0.115j   0.0356-0.115j
   0.0288-0.1009j  0.029 -0.0997j]
 [ 0.0143-0.076j   0.0052-0.0792j  0.004 -0.0792j  0.0042-0.0826j
   0.008 -0.0906j  0.0104-0.0881j]]

Top-left 6x6 of Hadd_ref (collated true) for antenna 0:
[[-0.0056+0.0827j -0.0443+0.0671j -0.0607+0.02

In [ ]:
import os
import math
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader

CKPT_PATH = "/kaggle/working/channel_est_preload_all_chunks.pth"  # checkpoint saved from training
DATA_DIR = DATA_DIR  
PHI_PATH = PHI_PATH 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
NUM_WORKERS = 4
PIN_MEMORY = True


ALPHA_PER_ANTENNA = 0.01   
USE_TRUE_FOR_TAU = False   
SAMPLE_TO_SAVE = 0      # index into validation split
ANTENNA_TO_SAVE = 0    
OUT_DIR = "/kaggle/working"   

os.makedirs(OUT_DIR, exist_ok=True)

ds_val = PreloadedSamplesFirstDataset(DATA_DIR, split='val', split_ratio=0.9, verbose=False)

Phi_raw = _load_mat_var(PHI_PATH, 'Phi')
Phi_tmp = _to_numpy_complex(Phi_raw)
if Phi_tmp.shape[0] == ds_val.numDD:
    Phi = Phi_tmp
elif Phi_tmp.shape[1] == ds_val.numDD:
    Phi = Phi_tmp.T.copy()
else:
    raise RuntimeError("Phi shape mismatch vs dataset.numDD")

pilot_rows, pilot_cols = derive_pilot_coords_from_Phi(Phi, ds_val.M, ds_val.N, ds_val.Nt)
collate_fn = make_batch_collate_fn(Phi, ds_val.M, ds_val.N, ds_val.Nt, pilot_rows, pilot_cols)

loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found at: {CKPT_PATH}")
ck = torch.load(CKPT_PATH, map_location=DEVICE)
if isinstance(ck, dict) and 'model_state' in ck:
    state = ck['model_state']
elif isinstance(ck, dict) and 'model_state_dict' in ck:
    state = ck['model_state_dict']
elif isinstance(ck, dict) and 'state_dict' in ck:
    state = ck['state_dict']
else:
    state = ck

model = ChannelEst2DNet(in_channels=2*ds_val.Nt, Nt=ds_val.Nt, M=ds_val.M, N=ds_val.N).to(DEVICE)
try:
    model.load_state_dict(state)
except Exception:
    new = {}
    for k,v in state.items():
        nk = k[len("module."): ] if k.startswith("module.") else k
        new[nk] = v
    model.load_state_dict(new)
model.eval()

def hard_threshold_per_antenna_batch(Hpred_batch, Htrue_batch=None, alpha_per_ant=0.01, use_true=False):
    
    B, M, N, Nt = Hpred_batch.shape
    if np.isscalar(alpha_per_ant):
        alpha_vec = np.ones(Nt, dtype=float) * float(alpha_per_ant)
    else:
        alpha_vec = np.asarray(alpha_per_ant, dtype=float)
        if alpha_vec.size != Nt:
            raise ValueError("alpha_per_ant must be scalar or array-like of length Nt")

    Hpred_thr = np.zeros_like(Hpred_batch)
    mask = np.zeros((B, M, N, Nt), dtype=bool)
    for b in range(B):
        for t in range(Nt):
            if use_true:
                if Htrue_batch is None:
                    raise ValueError("Htrue_batch required when use_true=True")
                base = np.abs(Htrue_batch[b, :, :, t])
            else:
                base = np.abs(Hpred_batch[b, :, :, t])
            max_val = base.max() if base.size > 0 else 0.0
            tau = alpha_vec[t] * float(max_val)
            keep_mask = (np.abs(Hpred_batch[b, :, :, t]) >= tau)
            mask[b, :, :, t] = keep_mask
            Hpred_thr[b, :, :, t] = Hpred_batch[b, :, :, t] * keep_mask
    return Hpred_thr, mask

total_err = 0.0
total_pow = 0.0
total_zeroed_locs = 0
total_locs = 0

with torch.no_grad():
    for Xv, Yv in loader_val:
        Xv = Xv.to(DEVICE); Yv = Yv.to(DEVICE)
        Ypv = model(Xv)                       # (B, 2*Nt, M, N)
        Ypv_np = Ypv.cpu().numpy()
        Yv_np  = Yv.cpu().numpy()
        B, C, Mm, Nn = Yv_np.shape
        Nt_here = C // 2
        Ypv_c = Ypv_np[:, :Nt_here, ...] + 1j * Ypv_np[:, Nt_here:, ...]
        Yv_c  = Yv_np[:, :Nt_here, ...]  + 1j * Yv_np[:, Nt_here:, ...]
        Ypv_c = np.transpose(Ypv_c, (0, 2, 3, 1))
        Yv_c  = np.transpose(Yv_c,  (0, 2, 3, 1))
        Ypv_thr, mask = hard_threshold_per_antenna_batch(Ypv_c, Htrue_batch=Yv_c, alpha_per_ant=ALPHA_PER_ANTENNA, use_true=USE_TRUE_FOR_TAU)
        err_batch = np.sum(np.abs(Ypv_thr - Yv_c)**2)
        pow_batch = np.sum(np.abs(Yv_c)**2)
        total_err += float(err_batch)
        total_pow += float(pow_batch)
        total_zeroed_locs += int(np.sum(~mask))
        total_locs += mask.size

if total_pow == 0:
    print("Warning: total power in validation set is zero. NMSE undefined.")
else:
    nmse_linear = total_err / total_pow
    nmse_db = 10.0 * math.log10(nmse_linear)
    print(f"Validation NMSE (after per-antenna hard thresholding): {nmse_linear:.6e}  ({nmse_db:.3f} dB)")

pct_zeroed = 100.0 * total_zeroed_locs / float(total_locs) if total_locs>0 else 0.0
print(f"Thresholding zeroed {total_zeroed_locs} / {total_locs} locations across val set ({pct_zeroed:.3f}%)")

if SAMPLE_TO_SAVE < 0 or SAMPLE_TO_SAVE >= len(ds_val):
    raise ValueError(f"SAMPLE_TO_SAVE must be in [0, {len(ds_val)-1}]")
y_sample, Hadd_true_sample = ds_val[SAMPLE_TO_SAVE]  # Hadd_true_sample shape (M,N,Nt)
X_batch_single, Y_batch_single = collate_fn([(y_sample, Hadd_true_sample)])
with torch.no_grad():
    Yp_single = model(X_batch_single.to(DEVICE)).cpu().numpy()
B_s, C_s, M_s, N_s = Yp_single.shape
Nt_s = C_s // 2
Yp_complex = Yp_single[0, :Nt_s, ...] + 1j * Yp_single[0, Nt_s:, ...]  # (Nt, M, N)
Hadd_cap_single = np.transpose(Yp_complex, (1,2,0))  # (M,N,Nt)
Hadd_cap_single_thr, mask_single = hard_threshold_per_antenna_batch(Hadd_cap_single[None,...], Htrue_batch=Hadd_true_sample[None,...],
                                                                   alpha_per_ant=ALPHA_PER_ANTENNA, use_true=USE_TRUE_FOR_TAU)
Hadd_cap_single_thr = Hadd_cap_single_thr[0]  # (M,N,Nt)

ant = ANTENNA_TO_SAVE
if ant < 0 or ant >= ds_val.Nt:
    raise ValueError(f"ANTENNA_TO_SAVE must be in [0, {ds_val.Nt-1}]")
Hcap_ant = Hadd_cap_single[:, :, ant]
Hcap_ant_thr = Hadd_cap_single_thr[:, :, ant]
Htrue_ant = Hadd_true_sample[:, :, ant]

def complex_to_str_grid(mat):
    Mloc, Nloc = mat.shape
    grid = np.empty((Mloc, Nloc), dtype=object)
    for i in range(Mloc):
        for j in range(Nloc):
            re = float(mat[i,j].real); im = float(mat[i,j].imag)
            sign = '+' if im >= 0 else '-'
            grid[i,j] = f"{re:.6e}{sign}{abs(im):.6e}j"
    return grid

cap_grid = complex_to_str_grid(Hcap_ant)
cap_thr_grid = complex_to_str_grid(Hcap_ant_thr)
true_grid = complex_to_str_grid(Htrue_ant)

cap_path = os.path.join(OUT_DIR, f"Hcap_val_sample{SAMPLE_TO_SAVE}_ant{ant}.csv")
cap_thr_path = os.path.join(OUT_DIR, f"Hcap_thr_val_sample{SAMPLE_TO_SAVE}_ant{ant}.csv")
true_path = os.path.join(OUT_DIR, f"Htrue_val_sample{SAMPLE_TO_SAVE}_ant{ant}.csv")

pd.DataFrame(cap_grid).to_csv(cap_path, index=False, header=False)
pd.DataFrame(cap_thr_grid).to_csv(cap_thr_path, index=False, header=False)
pd.DataFrame(true_grid).to_csv(true_path, index=False, header=False)

print(f"Saved predicted CSV: {cap_path}")
print(f"Saved predicted-thresholded CSV: {cap_thr_path}")
print(f"Saved true CSV:      {true_path}")

Validation NMSE (after per-antenna hard thresholding): 4.219530e-02  (-13.747 dB)
Thresholding zeroed 124798399 / 131072000 locations across val set (95.214%)
Saved predicted CSV: /kaggle/working/Hcap_val_sample0_ant0.csv
Saved predicted-thresholded CSV: /kaggle/working/Hcap_thr_val_sample0_ant0.csv
Saved true CSV:      /kaggle/working/Htrue_val_sample0_ant0.csv


In [ ]:


import os, glob, re, time, math
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset

CKPT_PATH = "/kaggle/input/otfs-channel-estiamtion-v3/channel_est_preload_all_chunks.pth"   # trained checkpoint (from your training cell)
NPY_CHUNKS_DIR = "/kaggle/input/dataset-npy-variable-snr/dataset-npy-variable-snr"  # folder containing chunk_snrXX_HADD.npy / _yDD.npy
PHI_PATH = "/kaggle/input/phi-256-8/Phi_pilotWeighted_padded.mat"
OUT_DIR = "/kaggle/working/eval_by_snr"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 64
NUM_WORKERS = 2
PIN_MEMORY = False

ALPHA_PER_ANTENNA = 0.01   # per-antenna hard threshold
USE_TRUE_FOR_TAU = False  

os.makedirs(OUT_DIR, exist_ok=True)

class NpyChunkDataset(Dataset):
    def __init__(self, y_memmap, hadd_memmap):
        self.y = y_memmap
        self.hadd = hadd_memmap
        if self.y.shape[0] != self.hadd.shape[0]:
            raise ValueError("y and Hadd must have same first dim (num samples)")
    def __len__(self):
        return int(self.y.shape[0])
    def __getitem__(self, idx):
        y_row = self.y[idx].astype(np.complex64)
        hadd_sample = self.hadd[idx].astype(np.complex64)   
        
        return y_row, hadd_sample

def hard_threshold_per_antenna_batch(Hpred_batch, Htrue_batch=None, alpha_per_ant=0.01, use_true=False):
    B, M, N, Nt = Hpred_batch.shape
    if np.isscalar(alpha_per_ant):
        alpha_vec = np.ones(Nt, dtype=float) * float(alpha_per_ant)
    else:
        alpha_vec = np.asarray(alpha_per_ant, dtype=float)
        if alpha_vec.size != Nt:
            raise ValueError("alpha_per_ant must be scalar or array-like of length Nt")
    Hpred_thr = np.zeros_like(Hpred_batch)
    mask = np.zeros((B, M, N, Nt), dtype=bool)
    for b in range(B):
        for t in range(Nt):
            if use_true:
                if Htrue_batch is None:
                    raise ValueError("Htrue_batch required when use_true=True")
                base = np.abs(Htrue_batch[b, :, :, t])
            else:
                base = np.abs(Hpred_batch[b, :, :, t])
            max_val = base.max() if base.size > 0 else 0.0
            tau = alpha_vec[t] * float(max_val)
            keep_mask = (np.abs(Hpred_batch[b, :, :, t]) >= tau)
            mask[b, :, :, t] = keep_mask
            Hpred_thr[b, :, :, t] = Hpred_batch[b, :, :, t] * keep_mask
    return Hpred_thr, mask


if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")
ck = torch.load(CKPT_PATH, map_location=DEVICE)
if isinstance(ck, dict) and 'model_state' in ck:
    state = ck['model_state']
elif isinstance(ck, dict) and 'model_state_dict' in ck:
    state = ck['model_state_dict']
elif isinstance(ck, dict) and 'state_dict' in ck:
    state = ck['state_dict']
else:
    state = ck

hadd_files = sorted(glob.glob(os.path.join(NPY_CHUNKS_DIR, "*_HADD.npy")))
pairs = []
for h in hadd_files:
    base = os.path.basename(h).replace("_HADD.npy", "")
    y = os.path.join(NPY_CHUNKS_DIR, base + "_yDD.npy")
    if os.path.exists(y):
        pairs.append((base, h, y))
    else:
        print("Warning: missing y pair for", h)

if len(pairs) == 0:
    raise RuntimeError("No chunk pairs found in NPY_CHUNKS_DIR")

Phi_raw = _load_mat_var(PHI_PATH, 'Phi')
Phi_tmp = _to_numpy_complex(Phi_raw)

summary_rows = []
model = None
model_created = False

for base, hadd_path, y_path in pairs:
    print("\n" + "="*70)
    print("Processing chunk:", base)
    t0 = time.time()

    hadd_mm = np.load(hadd_path, mmap_mode='r')   
    y_mm    = np.load(y_path, mmap_mode='r')     

    P = int(hadd_mm.shape[0])
    if hadd_mm.ndim != 4:
        raise RuntimeError(f"Unexpected hadd shape {hadd_mm.shape} for {hadd_path}")
    if hadd_mm.shape[1] <= 64 and hadd_mm.shape[3] > 20:  # likely (P, Nt, N, M)
        Nt = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); M = int(hadd_mm.shape[3])
        hadd_layout = 'P,Nt,N,M'
    else:
        if hadd_mm.shape[3] <= 64:
            M = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); Nt = int(hadd_mm.shape[3])
            hadd_layout = 'P,M,N,Nt'
        else:
            Nt = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); M = int(hadd_mm.shape[3])
            hadd_layout = 'P,Nt,N,M'
    print(f" chunk samples P={P}, inferred M={M}, N={N}, Nt={Nt}  (layout guess: {hadd_layout})")

    Phi_local = Phi_tmp
    numDD = M * N
    if Phi_local.shape[0] == numDD:
        Phi = Phi_local
    elif Phi_local.shape[1] == numDD:
        Phi = Phi_local.T.copy()
        print(" Transposed Phi to match numDD")
    else:
        try:
            y_numDD = int(y_mm.shape[1])
            if Phi_local.shape[0] == y_numDD:
                Phi = Phi_local
            elif Phi_local.shape[1] == y_numDD:
                Phi = Phi_local.T.copy()
                print(" Transposed Phi to match y's numDD axis")
            else:
                print(" Warning: could not confidently orient Phi vs M*N; proceeding with Phi as loaded")
                Phi = Phi_local
        except Exception:
            Phi = Phi_local

    pilot_rows, pilot_cols = derive_pilot_coords_from_Phi(Phi, M, N, Nt)
    collate_fn = make_batch_collate_fn(Phi, M, N, Nt, pilot_rows, pilot_cols)

    class _WrapDataset(Dataset):
        def __init__(self, y_mem, hadd_mem, layout):
            self.y = y_mem
            self.hadd = hadd_mem
            self.layout = layout
        def __len__(self): return int(self.y.shape[0])
        def __getitem__(self, idx):
            yrow = self.y[idx].astype(np.complex64)
            hadd = self.hadd[idx].astype(np.complex64)
            if self.layout == 'P,Nt,N,M':
                Hadd_out = np.transpose(hadd, (2,1,0)).astype(np.complex64)
            elif self.layout == 'P,M,N,Nt':
                Hadd_out = hadd.astype(np.complex64)
            else:
                Hadd_out = np.transpose(hadd, (2,1,0)).astype(np.complex64)
            return yrow, Hadd_out

    ds = _WrapDataset(y_mm, hadd_mm, hadd_layout)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=PIN_MEMORY, collate_fn=collate_fn)

    if not model_created:
        model = ChannelEst2DNet(in_channels=2*Nt, Nt=Nt, M=M, N=N).to(DEVICE)
        try:
            model.load_state_dict(state)
        except Exception:
            new = {}
            for k,v in state.items():
                nk = k[len("module."): ] if k.startswith("module.") else k
                new[nk] = v
            model.load_state_dict(new)
        model.eval()
        model_created = True
    else:
        pass

    total_err_raw = 0.0
    total_err_thr = 0.0
    total_pow = 0.0
    total_zeroed_locs = 0
    total_locs = 0
    processed = 0

    with torch.no_grad():
        for Xb, Yb in loader:
            Xb = Xb.to(DEVICE); Yb = Yb.to(DEVICE)
            Yp = model(Xb)   # (B, 2*Nt, M, N)

            
            Yp_np = Yp.cpu().numpy()
            Y_np  = Yb.cpu().numpy()
            B, C, Mm, Nn = Y_np.shape
            Nt_here = C // 2
            Yp_c = Yp_np[:, :Nt_here, ...] + 1j * Yp_np[:, Nt_here:, ...]
            Y_c  = Y_np[:, :Nt_here, ...]  + 1j * Y_np[:, Nt_here:, ...]
            Yp_c = np.transpose(Yp_c, (0, 2, 3, 1))
            Y_c  = np.transpose(Y_c,  (0, 2, 3, 1))

            err_raw = np.sum(np.abs(Yp_c - Y_c)**2)
            pow_batch = np.sum(np.abs(Y_c)**2)

            # threshold
            Yp_thr, mask = hard_threshold_per_antenna_batch(Yp_c, Htrue_batch=Y_c,
                                                           alpha_per_ant=ALPHA_PER_ANTENNA,
                                                           use_true=USE_TRUE_FOR_TAU)
            err_thr = np.sum(np.abs(Yp_thr - Y_c)**2)

            total_err_raw += float(err_raw)
            total_err_thr += float(err_thr)
            total_pow += float(pow_batch)
            total_zeroed_locs += int(np.sum(~mask))
            total_locs += mask.size
            processed += Yp_c.shape[0]

    mse_before = total_err_raw / float(total_locs) if total_locs>0 else float('nan')
    mse_after  = total_err_thr / float(total_locs) if total_locs>0 else float('nan')
    nmse_before = (total_err_raw / total_pow) if total_pow>0 else float('nan')
    nmse_after  = (total_err_thr / total_pow) if total_pow>0 else float('nan')
    nmse_before_db = 10.0 * math.log10(nmse_before) if (total_pow>0 and nmse_before>0) else float('nan')
    nmse_after_db  = 10.0 * math.log10(nmse_after)  if (total_pow>0 and nmse_after>0) else float('nan')
    pct_zeroed = 100.0 * total_zeroed_locs / float(total_locs) if total_locs>0 else 0.0

    print(f"Chunk {base}: processed samples={processed}, time={time.time()-t0:.1f}s")
    print(f"  MSE before: {mse_before:.6e}, after: {mse_after:.6e}")
    print(f"  NMSE before: {nmse_before:.6e} ({nmse_before_db:.3f} dB), after: {nmse_after:.6e} ({nmse_after_db:.3f} dB)")
    print(f"  Zeroed: {total_zeroed_locs}/{total_locs} ({pct_zeroed:.3f}%)")

    m = re.search(r"snr[_\-]?(\d+)", base, flags=re.IGNORECASE)
    snr_num = int(m.group(1)) if m else None

    summary_rows.append({
        "chunk": base,
        "snr_label": base,
        "snr_numeric": snr_num,
        "num_samples": processed,
        "mse_before": mse_before,
        "mse_after": mse_after,
        "nmse_before": nmse_before,
        "nmse_after": nmse_after,
        "nmse_before_db": nmse_before_db,
        "nmse_after_db": nmse_after_db,
        "pct_zeroed": pct_zeroed
    })

summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(OUT_DIR, "eval_summary_by_chunk.csv")
summary_df.to_csv(summary_csv, index=False)
print("\nSaved chunk summary to:", summary_csv)
print(summary_df)

if 'snr_numeric' in summary_df.columns:
    grp = summary_df.groupby('snr_numeric').agg({
        'num_samples': 'sum',
        'mse_before': 'mean',
        'mse_after': 'mean',
        'nmse_before': 'mean',
        'nmse_after': 'mean',
        'nmse_before_db': 'mean',
        'nmse_after_db': 'mean',
        'pct_zeroed': 'mean'
    }).reset_index().sort_values('snr_numeric')
    grp_csv = os.path.join(OUT_DIR, "eval_summary_by_snr.csv")
    grp.to_csv(grp_csv, index=False)
    print("\nAggregated by numeric SNR saved to:", grp_csv)
    print(grp)
else:
    print("No numeric SNR labels parsed; look at eval_summary_by_chunk.csv")


Processing chunk: chunk_snr05
 chunk samples P=5000, inferred M=256, N=8, Nt=16  (layout guess: P,Nt,N,M)
 Transposed Phi to match numDD
Chunk chunk_snr05: processed samples=5000, time=19.0s
  MSE before: 1.132075e-04, after: 1.133332e-04
  NMSE before: 5.934669e-02 (-12.266 dB), after: 5.941257e-02 (-12.261 dB)
  Zeroed: 155787242/163840000 (95.085%)

Processing chunk: chunk_snr10
 chunk samples P=5000, inferred M=256, N=8, Nt=16  (layout guess: P,Nt,N,M)
 Transposed Phi to match numDD
Chunk chunk_snr10: processed samples=5000, time=18.6s
  MSE before: 8.251505e-05, after: 8.268387e-05
  NMSE before: 4.321697e-02 (-13.643 dB), after: 4.330540e-02 (-13.635 dB)
  Zeroed: 155906324/163840000 (95.158%)

Processing chunk: chunk_snr15
 chunk samples P=5000, inferred M=256, N=8, Nt=16  (layout guess: P,Nt,N,M)
 Transposed Phi to match numDD
Chunk chunk_snr15: processed samples=5000, time=18.5s
  MSE before: 7.215649e-05, after: 7.232390e-05
  NMSE before: 3.765921e-02 (-14.241 dB), after: 3

In [8]:
!ls /kaggle/input/otfs-channel-estiamtion-v3

channel_est_preload_all_chunks.pth  Htrue_sample0_ant0.csv
custom.css			    Htrue_val_sample0_ant0.csv
eval_by_snr			    __notebook__.ipynb
Hcap_sample0_ant0.csv		    __output__.json
Hcap_thr_val_sample0_ant0.csv	    __results__.html
Hcap_val_sample0_ant0.csv


In [ ]:
import time, math, os, glob, numpy as np, torch
from torch.utils.data import Dataset

def measure_end2end_latency_for_chunk(hadd_path, y_path, ckpt_path, Phi_tmp,
                                      sample_idx=0, repeats=500, warmup=20,
                                      include_device_copy=True, device=None,
                                      verbose=True):
    """
    Measures: collate_time (CPU), copy_time (host->device), model_time (device),
              e2e_time (collate+copy+model).
    Returns dict with arrays and aggregated stats.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    hadd_mm = np.load(hadd_path, mmap_mode='r')
    y_mm    = np.load(y_path, mmap_mode='r')
    if hadd_mm.ndim != 4:
        raise RuntimeError(f"Unexpected hadd shape {hadd_mm.shape} for {hadd_path}")
    if hadd_mm.shape[1] <= 64 and hadd_mm.shape[3] > 20:
        Nt = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); M = int(hadd_mm.shape[3])
        hadd_layout = 'P,Nt,N,M'
    else:
        if hadd_mm.shape[3] <= 64:
            M = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); Nt = int(hadd_mm.shape[3])
            hadd_layout = 'P,M,N,Nt'
        else:
            Nt = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); M = int(hadd_mm.shape[3])
            hadd_layout = 'P,Nt,N,M'

    Phi_local = Phi_tmp
    numDD = M * N
    if Phi_local.shape[0] == numDD:
        Phi_used = Phi_local
    elif Phi_local.shape[1] == numDD:
        Phi_used = Phi_local.T.copy()
    else:
        Phi_used = Phi_local

    pilot_rows, pilot_cols = derive_pilot_coords_from_Phi(Phi_used, M, N, Nt)
    collate_fn = make_batch_collate_fn(Phi_used, M, N, Nt, pilot_rows, pilot_cols)

    def get_sample(idx):
        y_row = y_mm[idx].astype(np.complex64)
        hadd = hadd_mm[idx].astype(np.complex64)
        if hadd_layout == 'P,Nt,N,M':
            Hadd_out = np.transpose(hadd, (2,1,0)).astype(np.complex64)  # -> (M,N,Nt)
        elif hadd_layout == 'P,M,N,Nt':
            Hadd_out = hadd.astype(np.complex64)
        else:
            Hadd_out = np.transpose(hadd, (2,1,0)).astype(np.complex64)
        return y_row, Hadd_out

    if sample_idx < 0 or sample_idx >= int(y_mm.shape[0]):
        raise ValueError("sample_idx out of range for this chunk")

    ck = torch.load(ckpt_path, map_location=device)
    if isinstance(ck, dict) and 'model_state' in ck: state = ck['model_state']
    elif isinstance(ck, dict) and 'model_state_dict' in ck: state = ck['model_state_dict']
    elif isinstance(ck, dict) and 'state_dict' in ck: state = ck['state_dict']
    else: state = ck
    model = ChannelEst2DNet(in_channels=2*Nt, Nt=Nt, M=M, N=N).to(device)
    try:
        model.load_state_dict(state)
    except Exception:
        new = {}
        for k,v in state.items():
            nk = k[len("module."): ] if k.startswith("module.") else k
            new[nk] = v
        model.load_state_dict(new)
    model.eval()

    y_row, Hadd = get_sample(sample_idx)

    for _ in range(5):
        _ = collate_fn([(y_row, Hadd)])   # CPU ops
    if device.startswith('cuda'):
        torch.cuda.synchronize()

    X_cpu_once, _ = collate_fn([(y_row, Hadd)])  
    X_dev_once = X_cpu_once.to(device)           

    with torch.no_grad():
        for _ in range(warmup):
            Xc, _ = collate_fn([(y_row, Hadd)])
            if include_device_copy:
                Xd = Xc.to(device)
            else:
                Xd = Xc.to(device)  # still move to device; optional
            _ = model(Xd)
        if device.startswith('cuda'):
            torch.cuda.synchronize()

    collate_times = np.zeros(repeats, dtype=float)
    copy_times    = np.zeros(repeats, dtype=float)
    model_times   = np.zeros(repeats, dtype=float)
    e2e_times     = np.zeros(repeats, dtype=float)

    use_cuda_events = device.startswith('cuda')

    for r in range(repeats):
        t0 = time.perf_counter()
        Xc, _ = collate_fn([(y_row, Hadd)])   # CPU tensor (float32)
        t1 = time.perf_counter()
        collate_times[r] = t1 - t0

        if include_device_copy:
            if use_cuda_events:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            Xd = Xc.to(device)
            if use_cuda_events:
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            copy_times[r] = t1 - t0
        else:
            copy_times[r] = 0.0
            Xd = Xc.to(device)  

        if use_cuda_events:
            starter = torch.cuda.Event(enable_timing=True)
            ender   = torch.cuda.Event(enable_timing=True)
            torch.cuda.synchronize()
            starter.record()
            with torch.no_grad():
                _ = model(Xd)
            ender.record()
            torch.cuda.synchronize()
            ms = starter.elapsed_time(ender)  
            model_times[r] = ms * 1e-3    
        else:
            t0 = time.perf_counter()
            with torch.no_grad():
                _ = model(Xd)
            t1 = time.perf_counter()
            model_times[r] = t1 - t0

        t0 = time.perf_counter()
        Xc2, _ = collate_fn([(y_row, Hadd)])
        Xd2 = Xc2.to(device)
        with torch.no_grad():
            _ = model(Xd2)
        if device.startswith('cuda'):
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        e2e_times[r] = t1 - t0

    def stats(arr):
        return {'mean': float(np.mean(arr)),
                'median': float(np.median(arr)),
                'std': float(np.std(arr)),
                'min': float(np.min(arr))}

    results = {
        'collate_times': collate_times,
        'copy_times'   : copy_times,
        'model_times'  : model_times,
        'e2e_times'    : e2e_times,
        'collate_stats': stats(collate_times),
        'copy_stats'   : stats(copy_times),
        'model_stats'  : stats(model_times),
        'e2e_stats'    : stats(e2e_times),
        'meta': {
            'chunk_basename': os.path.basename(hadd_path),
            'hadd_layout': hadd_layout,
            'Nt': Nt, 'M': M, 'N': N,
            'device': device,
            'repeats': repeats,
            'warmup': warmup,
            'include_device_copy': include_device_copy
        }
    }

    if verbose:
        print("Measured end-to-end latency (per-sample) over {} repeats:".format(repeats))
        print("  Collate (CPU) mean = {mean:.6e}s  median = {median:.6e}s  std = {std:.6e}s".format(**results['collate_stats']))
        print("  Copy   (H->D) mean = {mean:.6e}s  median = {median:.6e}s  std = {std:.6e}s".format(**results['copy_stats']))
        print("  Model  (device) mean = {mean:.6e}s  median = {median:.6e}s  std = {std:.6e}s".format(**results['model_stats']))
        print("  E2E    (all)    mean = {mean:.6e}s  median = {median:.6e}s  std = {std:.6e}s".format(**results['e2e_stats']))

    return results


hadd_files = sorted(glob.glob(os.path.join(NPY_CHUNKS_DIR, "*_HADD.npy")))
pairs = []
for h in hadd_files:
    base = os.path.basename(h).replace("_HADD.npy", "")
    y = os.path.join(NPY_CHUNKS_DIR, base + "_yDD.npy")
    if os.path.exists(y):
        pairs.append((base, h, y))
if len(pairs) == 0:
    raise RuntimeError("No chunk pairs found")

base, hadd_path, y_path = pairs[0]
print("Timing chunk:", base)

Phi_raw = _load_mat_var(PHI_PATH, 'Phi')
Phi_tmp = _to_numpy_complex(Phi_raw)

res = measure_end2end_latency_for_chunk(hadd_path, y_path, CKPT_PATH, Phi_tmp,
                                       sample_idx=0, repeats=300, warmup=20,
                                       include_device_copy=True, device=DEVICE)


Timing chunk: chunk_snr05
Measured end-to-end latency (per-sample) over 300 repeats:
  Collate (CPU) mean = 4.862631e-02s  median = 4.841824e-02s  std = 1.022181e-03s
  Copy   (H->D) mean = 2.045476e-04s  median = 1.951485e-04s  std = 2.145710e-05s
  Model  (device) mean = 1.527423e-03s  median = 1.501008e-03s  std = 1.347106e-04s
  E2E    (all)    mean = 5.037524e-02s  median = 5.014398e-02s  std = 1.422405e-03s


In [ ]:
import torch, numpy as np, time, os, glob
from torch.utils.data import Dataset


def make_collate_fn_gpu(Phi_np, M, N, Nt, pilot_rows, pilot_cols, device=None):
    
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    Phi_t = torch.from_numpy(Phi_np).to(device=device)           # (numDD, Ptot) complex64
    if not torch.is_complex(Phi_t):
        Phi_t = Phi_t.to(dtype=torch.complex64)

    Ptot = Phi_t.shape[1]
    expected_full = Nt * M * N
    full_grid = (Ptot == expected_full)
    if not full_grid:
        pr = torch.from_numpy(pilot_rows.astype(np.int64)).to(device=device)  # (P,)
        pc = torch.from_numpy(pilot_cols.astype(np.int64)).to(device=device)  # (P,)
        idx = (pr * N + pc).to(dtype=torch.long)  # linear indices (P,)
    else:
        idx = None

    def collate_fn_gpu(batch):
        B = len(batch)
        Ys_np = np.stack([b[0] for b in batch], axis=1).astype(np.complex64)   # (numDD, B)
        Ys = torch.from_numpy(Ys_np).to(device=device)                          # (numDD, B), complex
        if not torch.is_complex(Ys):
            Ys = Ys.to(dtype=torch.complex64)

        Feats = torch.matmul(Phi_t.conj().transpose(0,1), Ys)   # (Ptot, B) complex64

        if full_grid:
            Xc = Feats.T.reshape((B, Nt, M, N)).contiguous()
        else:
            P = Ptot // Nt
            feat3 = Feats.view(Nt, P, B).permute(2, 0, 1).contiguous()   # (B, Nt, P)
            X_flat = torch.zeros((B, Nt, M * N), dtype=torch.complex64, device=device)
            idx_expand = idx.view(1, 1, -1).expand(B, Nt, -1)   # (B, Nt, P)
            X_flat = X_flat.scatter(2, idx_expand, feat3)
            Xc = X_flat.view(B, Nt, M, N).contiguous()

        X_real = Xc.real.to(dtype=torch.float32)
        X_imag = Xc.imag.to(dtype=torch.float32)
        X_batch = torch.cat([X_real, X_imag], dim=1)   # (B, 2*Nt, M, N) float32 on device

        Hs_np = np.stack([b[1] for b in batch], axis=0).astype(np.complex64)  # (B, M, N, Nt)
        Hs_np = np.transpose(Hs_np, (0, 3, 1, 2)).copy()
        Hs_t = torch.from_numpy(Hs_np).to(device=device)
        if not torch.is_complex(Hs_t):
            Hs_t = Hs_t.to(dtype=torch.complex64)
        Y_real = Hs_t.real.to(dtype=torch.float32)
        Y_imag = Hs_t.imag.to(dtype=torch.float32)
        Y_batch = torch.cat([Y_real, Y_imag], dim=1)  

        return X_batch, Y_batch

    return collate_fn_gpu




def measure_end2end_latency_with_gpu_collate(hadd_path, y_path, ckpt_path, PHI_PATH,
                                             sample_idx=0, repeats=300, warmup=20,
                                             device=None, verbose=True):
    
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    Phi_raw = _load_mat_var(PHI_PATH, 'Phi')
    Phi_np = _to_numpy_complex(Phi_raw)   # numpy complex64

    hadd_mm = np.load(hadd_path, mmap_mode='r')
    y_mm    = np.load(y_path, mmap_mode='r')
    if hadd_mm.ndim != 4:
        raise RuntimeError(f"Unexpected hadd shape {hadd_mm.shape} for {hadd_path}")

    if hadd_mm.shape[1] <= 64 and hadd_mm.shape[3] > 20:
        Nt = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); M = int(hadd_mm.shape[3])
        hadd_layout = 'P,Nt,N,M'
    else:
        if hadd_mm.shape[3] <= 64:
            M = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); Nt = int(hadd_mm.shape[3])
            hadd_layout = 'P,M,N,Nt'
        else:
            Nt = int(hadd_mm.shape[1]); N = int(hadd_mm.shape[2]); M = int(hadd_mm.shape[3])
            hadd_layout = 'P,Nt,N,M'

    numDD = M * N
    if Phi_np.shape[0] == numDD:
        Phi_use = Phi_np
    elif Phi_np.shape[1] == numDD:
        Phi_use = Phi_np.T.copy()
    else:
        Phi_use = Phi_np

    pilot_rows, pilot_cols = derive_pilot_coords_from_Phi(Phi_use, M, N, Nt)
    collate_gpu = make_collate_fn_gpu(Phi_use, M, N, Nt, pilot_rows, pilot_cols, device=device)

    ck = torch.load(ckpt_path, map_location='cpu')
    if isinstance(ck, dict) and 'model_state' in ck:
        state = ck['model_state']
    elif isinstance(ck, dict) and 'model_state_dict' in ck:
        state = ck['model_state_dict']
    elif isinstance(ck, dict) and 'state_dict' in ck:
        state = ck['state_dict']
    else:
        state = ck

    model = ChannelEst2DNet(in_channels=2*Nt, Nt=Nt, M=M, N=N).to(device)
    try:
        model.load_state_dict(state)
    except Exception:
        new = {}
        for k,v in state.items():
            nk = k[len("module."): ] if k.startswith("module.") else k
            new[nk] = v
        model.load_state_dict(new)
    model.eval()

    if sample_idx < 0 or sample_idx >= int(y_mm.shape[0]):
        raise ValueError("sample_idx out of range")

    y_row = y_mm[sample_idx].astype(np.complex64)
    hadd_sample = hadd_mm[sample_idx].astype(np.complex64)
    if hadd_layout == 'P,Nt,N,M':
        Hadd_out = np.transpose(hadd_sample, (2,1,0)).astype(np.complex64)
    elif hadd_layout == 'P,M,N,Nt':
        Hadd_out = hadd_sample.astype(np.complex64)
    else:
        Hadd_out = np.transpose(hadd_sample, (2,1,0)).astype(np.complex64)

    batch = [(y_row, Hadd_out)]

    with torch.no_grad():
        for _ in range(warmup):
            Xb, Yb = collate_gpu(batch)    # collate on device
            _ = model(Xb)
        if device.startswith('cuda'):
            torch.cuda.synchronize()

    collate_times = np.zeros(repeats, dtype=float)
    model_times   = np.zeros(repeats, dtype=float)
    e2e_times     = np.zeros(repeats, dtype=float)

    use_cuda_events = device.startswith('cuda')

    for r in range(repeats):
        t0 = time.perf_counter()
        Xb, Yb = collate_gpu(batch)
        if use_cuda_events:
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        collate_times[r] = t1 - t0

        if use_cuda_events:
            start = torch.cuda.Event(enable_timing=True); end = torch.cuda.Event(enable_timing=True)
            torch.cuda.synchronize()
            start.record()
            with torch.no_grad():
                _ = model(Xb)
            end.record()
            torch.cuda.synchronize()
            model_times[r] = start.elapsed_time(end) * 1e-3   # ms -> s
        else:
            t0 = time.perf_counter()
            with torch.no_grad():
                _ = model(Xb)
            t1 = time.perf_counter()
            model_times[r] = t1 - t0

        t0 = time.perf_counter()
        Xb2, Yb2 = collate_gpu(batch)
        with torch.no_grad():
            _ = model(Xb2)
        if use_cuda_events:
            torch.cuda.synchronize()
        t1 = time.perf_counter()
        e2e_times[r] = t1 - t0

    def stats(a):
        return {'mean': float(a.mean()), 'median': float(np.median(a)), 'std': float(a.std()), 'min': float(a.min())}

    results = {
        'collate_times': collate_times, 'model_times': model_times, 'e2e_times': e2e_times,
        'collate_stats': stats(collate_times),
        'model_stats'  : stats(model_times),
        'e2e_stats'    : stats(e2e_times),
        'meta': {'hadd_path': hadd_path, 'sample_idx': sample_idx, 'device': device, 'Nt': Nt, 'M': M, 'N': N}
    }

    if verbose:
        print(f"Chunk: {os.path.basename(hadd_path)} | device={device} | sample={sample_idx}")
        print("  Collate (GPU) mean = {mean:.6e}s  median = {median:.6e}s  std = {std:.6e}s".format(**results['collate_stats']))
        print("  Model   (GPU) mean = {mean:.6e}s  median = {median:.6e}s  std = {std:.6e}s".format(**results['model_stats']))
        print("  E2E     (all) mean = {mean:.6e}s  median = {median:.6e}s  std = {std:.6e}s".format(**results['e2e_stats']))

    return results


hadd_files = sorted(glob.glob(os.path.join(NPY_CHUNKS_DIR, "*_HADD.npy")))
pairs = []
for h in hadd_files:
    base = os.path.basename(h).replace("_HADD.npy", "")
    y = os.path.join(NPY_CHUNKS_DIR, base + "_yDD.npy")
    if os.path.exists(y):
        pairs.append((base, h, y))
if len(pairs) == 0:
    raise RuntimeError("No chunk pairs found")

base, hadd_path, y_path = pairs[0]
res_gpu = measure_end2end_latency_with_gpu_collate(hadd_path, y_path, CKPT_PATH, PHI_PATH,
                                                   sample_idx=0, repeats=200, warmup=20,
                                                   device=DEVICE, verbose=True)


Chunk: chunk_snr05_HADD.npy | device=cuda | sample=0
  Collate (GPU) mean = 1.164553e-03s  median = 1.045184e-03s  std = 2.601164e-04s
  Model   (GPU) mean = 1.157760e-03s  median = 1.130496e-03s  std = 6.431301e-05s
  E2E     (all) mean = 2.294889e-03s  median = 2.181241e-03s  std = 3.178685e-04s
